# Prompt activation PCA: KMeans and HDDC

Fit 15 PCs per layer, then compare saved KMeans PCA geometry and HDDC Gaussians
in two independent K3D plots, each in its own cell with its own controls.
Both use the same PCA coordinates and ground-truth point colors.
PCA centers features without scaling or whitening.

Set the activation and model paths in the first code cell, then run cells in order.
To use years, set `ACTIVATION_DIR` and `MODEL_ROOT` to
`dalg-cache/sae_manifold_years_neutral_llama31_8b_last` and `SUBSET = "years"`.
Changing the dataset or subset requires rerunning PCA and model preparation.


In [1]:
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import json
import os
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
import k3d
from sklearn.decomposition import PCA
from sklearn.metrics import homogeneity_score
from joblib import Parallel, delayed, parallel_config
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from dalg.data.prompt_activations import load_prompt_activation_index
from dalg.data.shard_activations import ActivationBatchDataset
from dalg.models.kmeans import load_kmeans
from dalg.models.adaptive_q.mfa_hddc import load_mfa_hddc

# Set the activation and model paths for this analysis.
ACTIVATION_DIR = REPO_ROOT / "dalg-cache/sae_manifold_temperature_02d_llama31_8b_last"
MODEL_ROOT = REPO_ROOT / "dalg-cache/sae_manifold_temperature_02d_llama31_8b_last"
MODEL_ROOTS = [
    REPO_ROOT / "dalg-cache/sae_manifold_temperature_02d_llama31_8b_last",
    REPO_ROOT / "dalg-cache/sae_manifold_years_neutral_llama31_8b_last",
]

HOMOGENEITY_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOMOGENEITY_BATCH_SIZE = 512

config = json.loads((ACTIVATION_DIR / "config.json").read_text())
LAYERS = sorted(config["layers"])
# This reader checks completion and joins source records to activation coordinates.
index = load_prompt_activation_index(ACTIVATION_DIR, layer=LAYERS[0])
metadata = pd.DataFrame(index)
SUBSETS = sorted(metadata["subset"].unique())
summary = metadata.groupby("subset").size().rename("prompts").to_frame()
summary["label_fields"] = [
    sorted({key for labels in metadata.loc[metadata.subset.eq(name), "labels"]
            for key in labels})
    for name in summary.index
]
print(f"{config['model']}: {len(metadata):,} prompts, {len(LAYERS)} layers, "
      f"{config['d_model']} activation features; last token")
display(summary)

meta-llama/Llama-3.1-8B: 15,000 prompts, 32 layers, 4096 activation features; last token


,prompts,label_fields
subset,,
temperature,15000,"[celsius, fahrenheit]"


## Select a subset

Edit `SUBSET` to a name in the table above and rerun this cell and everything below.
`COLOR_LABEL = None` chooses a default ground-truth field. The K3D panels require
a finite numeric label, such as `fahrenheit` or `year`.
Changing only the coloring requires rerunning this cell and the explorer cell,
without recomputing PCA or model projections.


In [2]:
SUBSET = "temperature"  # colors, age, temperature, geography, days, years, formality, sent_length
COLOR_LABEL = None
SEED = 0
N_COMPONENTS = 15
N_WORKERS = min(8, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
                else (os.cpu_count() or 1))  # Increase to run more layers concurrently.

if SUBSET not in SUBSETS:
    raise ValueError(f"Choose SUBSET from {SUBSETS}")
selected = metadata.loc[metadata.subset.eq(SUBSET)].reset_index(drop=True)
labels = pd.json_normalize(selected["labels"].tolist())
DEFAULT_COLOR = {
    "colors": "hue", "age": "age", "temperature": "fahrenheit",
    "geography": "continent", "days": "day_idx", "years": "year",
    "formality": "formality", "sent_length": "n_tokens",
}
color_label = COLOR_LABEL or DEFAULT_COLOR.get(SUBSET, labels.columns[0])
if color_label not in labels:
    raise ValueError(f"Choose COLOR_LABEL from {labels.columns.tolist()}")
print(f"Selected {SUBSET}: {len(selected):,} prompts; color = {color_label}")
display(pd.concat([selected[["source_row_id", "prompt", "decoded_token"]], labels], axis=1).head())

Selected temperature: 15,000 prompts; color = fahrenheit


,source_row_id,prompt,decoded_token,fahrenheit,celsius
0,0,Today it's -30 degrees Fahrenheit outside,outside,-30.00,-34.444444
1,1,Today it's -30.01 degrees Fahrenheit outside,outside,-30.01,-34.450000
2,2,Today it's -30.02 degrees Fahrenheit outside,outside,-30.02,-34.455556
3,3,Today it's -30.03 degrees Fahrenheit outside,outside,-30.03,-34.461111
4,4,Today it's -30.04 degrees Fahrenheit outside,outside,-30.04,-34.466667


## Fit 15 PCs for every layer in parallel

`N_WORKERS` controls how many layers run concurrently (up to eight by default).
Each worker loads only the selected subset, uses one CPU thread, and fits all its
rows, including distinct records with duplicate prompt text. A seeded randomized
SVD estimates the top 15 PCs with 20 oversamples and 7 power iterations.

Completed layers report loading and PCA times immediately, in completion order.
The final tables and layer selector remain ordered by layer number. Each worker
holds one activation matrix, so memory use increases with `N_WORKERS`.
`results` retains scores, PCA models, and loading/fitting times. No extraction
artifacts are modified.

In [3]:
def load_subset_layer(rows, layer):
    """Read selected rows in their metadata order, grouping reads by saved shard."""
    groups = defaultdict(list)
    for output_row, row in enumerate(rows.itertuples(index=False)):
        groups[int(row.shard)].append((output_row, int(row.row_in_shard), int(row.token_slot)))
    values = np.empty((len(rows), config["d_model"]), dtype=np.float32)
    for shard_id, coordinates in groups.items():
        path = ACTIVATION_DIR / f"layer{layer:02d}" / f"shard_{shard_id:05d}.pt"
        shard = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
        if shard.ndim != 3 or tuple(shard.shape[1:]) != (1, config["d_model"]):
            raise ValueError(f"Unexpected activation shape in {path}: {tuple(shard.shape)}")
        output_rows, shard_rows, token_slots = map(list, zip(*coordinates))
        values[output_rows] = shard[shard_rows, token_slots].float().numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite activations in layer {layer}")
    return values


def fit_layer(layer, rows, seed):
    """Load and project one layer; joblib limits each worker to one CPU thread."""
    started = perf_counter()
    x = load_subset_layer(rows, layer)
    loaded = perf_counter()
    if not np.any(x != x[0]):
        raise ValueError(f"Layer {layer} has zero variance across the selected prompts")
    pca = PCA(n_components=N_COMPONENTS, svd_solver="randomized", random_state=seed,
              n_oversamples=20, iterated_power=7)
    scores = pca.fit_transform(x)
    fitted = perf_counter()
    result = {
        "scores": scores, "pca": pca,
        "load_seconds": loaded - started, "pca_seconds": fitted - loaded,
    }
    return layer, result


if min(len(selected) - 1, config["d_model"]) < N_COMPONENTS:
    raise ValueError(f"{N_COMPONENTS} centered PCs require at least {N_COMPONENTS + 1} prompts and {N_COMPONENTS} features")
if not isinstance(N_WORKERS, int) or N_WORKERS < 1:
    raise ValueError("N_WORKERS must be a positive integer")
results = {}
workers = min(N_WORKERS, len(LAYERS))
started = perf_counter()
print(f"Fitting {len(LAYERS)} layers with {workers} parallel workers...", flush=True)
# Separate processes avoid competing BLAS/OpenMP thread pools in the kernel.
with parallel_config(backend="loky", inner_max_num_threads=1):
    completed = Parallel(n_jobs=workers, return_as="generator_unordered")(
        delayed(fit_layer)(layer, selected, SEED) for layer in LAYERS
    )
    for layer, result in completed:
        results[layer] = result
        ratios = result["pca"].explained_variance_ratio_
        print(f"[{len(results):02d}/{len(LAYERS)}] Layer {layer:02d}: "
              f"top {N_COMPONENTS} explain {ratios.sum():.2%}; "
              f"load {result['load_seconds']:.2f}s, PCA {result['pca_seconds']:.2f}s",
              flush=True)
results = dict(sorted(results.items()))
variance = pd.DataFrame({
    layer: result["pca"].explained_variance_ratio_
    for layer, result in results.items()
}, index=[f"PC{i + 1}" for i in range(N_COMPONENTS)]).T
variance.index.name = "layer"
print(f"Finished in {perf_counter() - started:.1f}s", flush=True)

Fitting 32 layers with 8 parallel workers...
[01/32] Layer 04: top 15 explain 88.35%; load 0.45s, PCA 1.79s
[02/32] Layer 03: top 15 explain 88.17%; load 2.98s, PCA 2.39s
[03/32] Layer 02: top 15 explain 85.41%; load 3.31s, PCA 2.41s
[04/32] Layer 00: top 15 explain 84.11%; load 3.32s, PCA 2.54s
[05/32] Layer 01: top 15 explain 79.71%; load 3.46s, PCA 2.54s
[06/32] Layer 06: top 15 explain 93.10%; load 3.65s, PCA 2.33s
[07/32] Layer 07: top 15 explain 93.55%; load 3.23s, PCA 2.36s
[08/32] Layer 05: top 15 explain 91.96%; load 3.81s, PCA 2.35s
[09/32] Layer 08: top 15 explain 94.10%; load 3.94s, PCA 2.00s
[10/32] Layer 09: top 15 explain 93.71%; load 2.93s, PCA 2.16s
[11/32] Layer 12: top 15 explain 94.28%; load 2.77s, PCA 2.19s
[12/32] Layer 11: top 15 explain 93.84%; load 3.05s, PCA 2.18s
[13/32] Layer 13: top 15 explain 94.50%; load 3.16s, PCA 2.20s
[14/32] Layer 10: top 15 explain 93.84%; load 3.84s, PCA 2.13s
[15/32] Layer 14: top 15 explain 95.47%; load 3.42s, PCA 2.10s
[16/32] La

## Select saved cluster models

Set `MODEL_ROOT`, `MODEL_K`, and `SURGERY_THRESHOLD` in the first code cell.
Load both KMeans and HDDC using those settings.
Use the supplied model directory, selecting runs by layer, K, and threshold. The combined
table includes missing checkpoints; each panel reports its own missing models
and keeps showing points. If multiple runs match a model kind and layer, narrow
`MODEL_ROOT` rather than silently choosing a seed or checkpoint.
Rerun model preparation when these settings change.


## Component summary for all configurations

This table scans **all** KMeans and HDDC runs under `MODEL_ROOTS`, including both
datasets and every saved layer, K, threshold, and seed. It is independent of the
plot selection (`ACTIVATION_DIR`, `SUBSET`, `MODEL_K`, `SURGERY_THRESHOLD`) and does
not require fitted PCA. Run the first code cell first; rerun this
cell to refresh the table as checkpoints become available.

- **Alive / total:** HDDC components with finite mixture logits (nonzero model
  weight), or KMeans components with at least one training member. A live HDDC
  component may have `q_k = 0`; it is still a Gaussian with noise covariance.
- **mean_q_k:** unweighted mean saved rank among live components with valid rank
  estimates. For KMeans, clusters with fewer than two members have no valid PCA
  rank and are excluded. **rank_components** records the averaging denominator.
- Statistics come from the saved `mfa_model.pt` or `kmeans_model.pt` used by the
  plots, rather than the final training-history iteration. Missing checkpoints
  remain listed with unknown statistics, not zero alive components.

`configuration_summary` retains numeric counts and full run paths for filtering.

- **homogeneity:** label purity within clusters (0–1; higher is purer), using
  `fahrenheit` for temperature and `year` for years. Each exact value is a class,
  without binning. Scores use all prompts (training and validation), with
  nearest-centroid KMeans assignments or maximum-posterior HDDC assignments
  computed in the original activation space. Labels are joined by global row ID.
  Missing checkpoints have no score. The first cell sets the scoring device and
  batch size; the table takes longer because it now runs model inference.


In [4]:
def component_summary(model, kind):
    ranks = model.component_ranks.detach().cpu()
    if kind == "hddc":
        # Zero-weight EM components have -inf logits; avoid softmax underflow.
        alive = torch.isfinite(model.pi_logits.detach().cpu())
        rank_eligible = alive
    elif kind == "kmeans":
        alive = model.cluster_counts.detach().cpu() > 0
        rank_eligible = alive & model.pca_valid.detach().cpu()
    else:
        raise ValueError(f"Unsupported model kind: {kind}")
    rank_count = int(rank_eligible.sum())
    return {
        "components_alive": int(alive.sum()),
        "components_total": int(model.K),
        "mean_q_k": float(ranks[rank_eligible].double().mean()) if rank_count else np.nan,
        "rank_components": rank_count,
    }


def model_homogeneity(model, kind, activation_dir, layer, labels_by_row):
    """Score full-dataset assignments against labels aligned by global row ID."""
    stream = ActivationBatchDataset(
        activation_dir, layer, batch_size=HOMOGENEITY_BATCH_SIZE,
        return_metadata=True, shuffle_shards=False, shuffle_within_shard=False,
    )
    model.to(HOMOGENEITY_DEVICE).eval()
    cache = model.inference_cache() if kind == "hddc" else nullcontext()
    truth, assignments = [], []
    with torch.inference_mode(), cache:
        for x, global_rows, _ in stream:
            x = x.to(device=model.mu.device, dtype=model.mu.dtype)
            predicted = model.predict(x) if kind == "kmeans" else model.responsibilities(x).argmax(dim=1)
            assignments.extend(predicted.cpu().tolist())
            truth.extend(labels_by_row[int(row)] for row in global_rows)
    # Numeric temperatures and years are categorical labels, without binning.
    return homogeneity_score(pd.factorize(np.asarray(truth))[0], assignments)


def summarize_configurations(model_root):
    rows = []
    label_indexes = {}
    label_fields = {"temperature_02d": "fahrenheit", "years_neutral": "year"}
    for spec_path in sorted(model_root.rglob("run_spec.json")):
        spec = json.loads(spec_path.read_text())
        dataset, training = spec["dataset"], spec["training"]
        kind = training["model_kind"]
        if kind not in ("kmeans", "hddc"):
            continue
        args = training["arguments"]
        label_field = label_fields[dataset["id"]]
        checkpoint = spec_path.parent / ("mfa_model.pt" if kind == "hddc" else "kmeans_model.pt")
        row = {
            "dataset": dataset["id"], "layer": int(dataset["layer"]), "kind": kind,
            "components_total": int(args["K"]), "threshold": args["surgery_threshold"],
            "q_max": args.get("rank") if kind == "hddc" else None, "seed": args["seed"],
            "components_alive": None, "mean_q_k": np.nan, "rank_components": None,
            "checkpoint_status": "missing", "run_id": spec_path.parent.name,
            "checkpoint": str(checkpoint),
            "homogeneity": np.nan, "homogeneity_label": label_field,
        }
        if checkpoint.is_file():
            loader = load_mfa_hddc if kind == "hddc" else load_kmeans
            model = loader(checkpoint, map_location="cpu")
            row.update(component_summary(model, kind))
            activation_dir, layer = Path(dataset["shard_dir"]), int(dataset["layer"])
            label_key = (activation_dir, layer, label_field)
            if label_key not in label_indexes:
                prompt_index = load_prompt_activation_index(activation_dir, layer)
                label_indexes[label_key] = {
                    item["global_row"]: item["labels"][label_field] for item in prompt_index
                }
            row["homogeneity"] = model_homogeneity(
                model, kind, activation_dir, layer, label_indexes[label_key]
            )
            print(f"{spec_path.parent.name}: homogeneity={row['homogeneity']:.3f} ({label_field})", flush=True)
            row["checkpoint_status"] = "available"
            del model  # Keep at most one full checkpoint in memory.
        rows.append(row)
    columns = ["dataset", "layer", "kind", "components_total", "threshold", "q_max", "seed",
               "components_alive", "mean_q_k", "rank_components", "homogeneity", "homogeneity_label",
               "checkpoint_status", "run_id", "checkpoint"]
    table = pd.DataFrame(rows, columns=columns)
    for column in ("components_alive", "components_total", "rank_components", "q_max"):
        table[column] = table[column].astype("Int64")
    return table.sort_values(["dataset", "layer", "kind", "components_total", "threshold", "seed", "run_id"]).reset_index(drop=True)


configuration_summary = pd.concat(
    [summarize_configurations(root) for root in MODEL_ROOTS], ignore_index=True
)
configuration_table = configuration_summary.copy()
configuration_table["alive / total"] = [
    f"{int(alive) if pd.notna(alive) else '?'} / {int(total)}"
    for alive, total in zip(configuration_table["components_alive"], configuration_table["components_total"])
]
print(f"{len(configuration_summary)} configurations · "
      f"{configuration_summary['checkpoint_status'].eq('available').sum()} saved checkpoints")
display(configuration_table[[
    "dataset", "layer", "kind", "threshold", "q_max", "seed", "alive / total",
    "mean_q_k", "rank_components", "homogeneity", "homogeneity_label", "checkpoint_status", "run_id",
]].style.format({"mean_q_k": "{:.2f}", "homogeneity": "{:.3f}", "threshold": "{:g}"}, na_rep="—"))


hddc__temperature_02d__l04__k100__q16__s42__312db7b7: homogeneity=0.460 (fahrenheit)
hddc__temperature_02d__l04__k100__q16__s42__545b289f: homogeneity=0.462 (fahrenheit)
hddc__temperature_02d__l04__k100__q16__s42__9ce26c22: homogeneity=0.460 (fahrenheit)
hddc__temperature_02d__l04__k100__q16__s42__ece732d5: homogeneity=0.463 (fahrenheit)
hddc__temperature_02d__l04__k50__q16__s42__28ef0231: homogeneity=0.387 (fahrenheit)
hddc__temperature_02d__l04__k50__q16__s42__43eb60a9: homogeneity=0.387 (fahrenheit)
hddc__temperature_02d__l04__k50__q16__s42__a8bd7614: homogeneity=0.389 (fahrenheit)
hddc__temperature_02d__l04__k50__q16__s42__e76df1ed: homogeneity=0.389 (fahrenheit)
hddc__temperature_02d__l28__k100__q16__s42__655c7199: homogeneity=0.453 (fahrenheit)
hddc__temperature_02d__l28__k100__q16__s42__758e6b41: homogeneity=0.452 (fahrenheit)
hddc__temperature_02d__l28__k100__q16__s42__7eb4d680: homogeneity=0.456 (fahrenheit)
hddc__temperature_02d__l28__k100__q16__s42__de28e1e4: homogeneity=0.4

,dataset,layer,kind,threshold,q_max,seed,alive / total,mean_q_k,rank_components,homogeneity,homogeneity_label,checkpoint_status,run_id
0,temperature_02d,4,hddc,0.0001,16,42,50 / 50,16.00,50,0.389,fahrenheit,available,hddc__temperature_02d__l04__k50__q16__s42__a8bd7614
1,temperature_02d,4,hddc,0.001,16,42,50 / 50,15.94,50,0.389,fahrenheit,available,hddc__temperature_02d__l04__k50__q16__s42__e76df1ed
2,temperature_02d,4,hddc,0.01,16,42,50 / 50,12.22,50,0.387,fahrenheit,available,hddc__temperature_02d__l04__k50__q16__s42__28ef0231
3,temperature_02d,4,hddc,0.1,16,42,50 / 50,3.70,50,0.387,fahrenheit,available,hddc__temperature_02d__l04__k50__q16__s42__43eb60a9
4,temperature_02d,4,hddc,0.0001,16,42,100 / 100,15.78,100,0.463,fahrenheit,available,hddc__temperature_02d__l04__k100__q16__s42__ece732d5
5,temperature_02d,4,hddc,0.001,16,42,100 / 100,15.74,100,0.462,fahrenheit,available,hddc__temperature_02d__l04__k100__q16__s42__545b289f
6,temperature_02d,4,hddc,0.01,16,42,100 / 100,12.03,100,0.460,fahrenheit,available,hddc__temperature_02d__l04__k100__q16__s42__9ce26c22
7,temperature_02d,4,hddc,0.1,16,42,100 / 100,3.01,100,0.460,fahrenheit,available,hddc__temperature_02d__l04__k100__q16__s42__312db7b7
8,temperature_02d,4,kmeans,0.0001,—,42,50 / 50,190.50,50,0.379,fahrenheit,available,kmeans__temperature_02d__l04__k50__cattell__s42__5f399f1b
9,temperature_02d,4,kmeans,0.001,—,42,50 / 50,46.72,50,0.379,fahrenheit,available,kmeans__temperature_02d__l04__k50__cattell__s42__d430e0b1


## Shared model projection: first three local PCs

Each ellipsoid is defined by its component's **first three local covariance PCs**,
then projected into the selected global PCA coordinates. Local PC1/PC2/PC3 keep
these identities when changing the plotting axes.

For HDDC, obtain the leading covariance PCs from an SVD of its loadings, adding
the isotropic noise variance to their eigenvalues. Noise is retained along these
three PCs only. When leading eigenvalues tie (including noise-only directions),
the associated axes are not uniquely determined; the SVD selects a basis.
For KMeans, use the first three saved rank-selected PCs and their eigenvalues.
If fewer than three were saved, show only the available directions; do not invent
missing PCs. Invalid KMeans PCA components retain centers but no footprint.

Only the three scaled local PCs are projected into the fitted 15-PC space.
Models load sequentially, without constructing ambient covariance matrices.
Rerun this section after changing model settings or refitting global PCA.


In [5]:
MODEL_K = 100
SURGERY_THRESHOLD = 0.1

In [6]:
from scipy.stats import chi2


def discover_model_runs():
    matches = {kind: {} for kind in MODEL_KINDS}
    rows = []
    for spec_path in sorted(MODEL_ROOT.rglob("run_spec.json")):
        spec = json.loads(spec_path.read_text())
        training = spec["training"]
        kind, args = training["model_kind"], training["arguments"]
        layer = int(spec["dataset"]["layer"])
        if (kind not in matches or layer not in results or args["K"] != MODEL_K
                or not np.isclose(args["surgery_threshold"], SURGERY_THRESHOLD, rtol=1e-10, atol=0)):
            continue
        if layer in matches[kind]:
            raise ValueError(f"Multiple {kind} runs for layer {layer}; narrow MODEL_ROOT")
        filename = "mfa_model.pt" if kind == "hddc" else "kmeans_model.pt"
        checkpoint = spec_path.parent / filename
        matches[kind][layer] = checkpoint
        rows.append({"layer": layer, "kind": kind, "K": MODEL_K,
                     "threshold": SURGERY_THRESHOLD, "available": checkpoint.is_file(),
                     "run": spec_path.parent.name})
    table = pd.DataFrame(rows, columns=["layer", "kind", "K", "threshold", "available", "run"])
    return matches, table.sort_values(["layer", "kind"])


def leading_hddc_factors(loadings, psi):
    """Top three covariance PCs, scaled by sqrt(eigenvalue), for isotropic HDDC."""
    loadings = np.asarray(loadings, dtype=np.float64)
    psi = np.asarray(psi, dtype=np.float64)
    if not np.allclose(psi, psi[:, :1], rtol=1e-6, atol=0):
        raise ValueError("Local-PC visualization requires isotropic HDDC noise")
    if np.any(psi <= 0):
        raise ValueError("HDDC noise variances must be positive")
    count, dimension, _ = loadings.shape
    n_pcs = min(3, dimension)
    factors = np.zeros((count, dimension, 3), dtype=np.float64)
    for k, loading in enumerate(loadings):
        vectors, singular_values, _ = np.linalg.svd(loading, full_matrices=False)
        vectors = vectors[:, :n_pcs]
        # Small-q checkpoints may need noise-only PCs outside the loading span.
        for coordinate in range(dimension):
            if vectors.shape[1] == n_pcs:
                break
            direction = np.zeros(dimension)
            direction[coordinate] = 1
            for _ in range(2):
                direction -= vectors @ (vectors.T @ direction)
            norm = np.linalg.norm(direction)
            if norm > 1e-10:
                vectors = np.column_stack([vectors, direction / norm])
        values = np.full(n_pcs, psi[k, 0])
        n_signal = min(n_pcs, len(singular_values))
        values[:n_signal] += singular_values[:n_signal] ** 2
        factors[k, :, :n_pcs] = vectors * np.sqrt(values)
    return factors


def load_projected_model(checkpoint, kind, pca):
    loader = load_mfa_hddc if kind == "hddc" else load_kmeans
    model = loader(checkpoint, map_location="cpu")
    def array(tensor):
        return tensor.detach().cpu().numpy().astype(np.float64)
    with torch.no_grad():
        means = (array(model.mu) - pca.mean_) @ pca.components_.astype(np.float64).T
        ranks = model.component_ranks.detach().cpu().numpy()
        if kind == "hddc":
            factors = leading_hddc_factors(array(model._W()), array(model._psi()))
            valid = np.ones(model.K, dtype=bool)
            weights = array(model.pi_logits.softmax(dim=0))
            details = [f"mixture weight={weight:.5g}" for weight in weights]
            description = "HDDC · first 3 local covariance PCs"
        else:
            n_pcs = min(3, model.q)
            values = array(model.eigenvalues[:, :n_pcs])
            if np.any(values < 0):
                raise ValueError("Negative saved KMeans eigenvalues")
            factors = np.zeros((model.K, model.D, 3), dtype=np.float64)
            factors[:, :, :n_pcs] = array(model.W[:, :, :n_pcs]) * np.sqrt(values)[:, None, :]
            valid = model.pca_valid.detach().cpu().numpy().copy()
            counts = model.cluster_counts.detach().cpu().numpy()
            details = [f"training cluster count={count:,}" for count in counts]
            description = "KMeans · first 3 available saved local PCs"
    local_ranks = (np.linalg.norm(factors, axis=1) > 0).sum(axis=1)
    projected_factors = np.einsum("pd,kdr->kpr", pca.components_.astype(np.float64), factors, optimize=True)
    covariance = projected_factors @ projected_factors.transpose(0, 2, 1)
    if not (np.isfinite(means).all() and np.isfinite(covariance).all()):
        raise ValueError(f"Nonfinite projected geometry: {checkpoint}")
    colors = k3d.helpers.map_colors(
        np.linspace(0.02, 0.98, model.K), k3d.colormaps.matplotlib_color_maps.Turbo, [0, 1]
    )
    hover = [f"component {k}<br>{kind}<br>saved rank={int(ranks[k])}<br>local PCs used={int(local_ranks[k])}<br>{details[k]}"
             + ("<br>no valid PCA footprint" if not valid[k] else "") for k in range(model.K)]
    return {"means": means, "covariances": covariance, "valid": valid,
            "local_pc_factors": projected_factors, "local_ranks": local_ranks,
            "colors": colors, "hover": hover, "description": description,
            "checkpoint": str(checkpoint)}


MODEL_KINDS = ("kmeans", "hddc")
matched_runs, run_table = discover_model_runs()
display(run_table)
cluster_geometry = {kind: {} for kind in MODEL_KINDS}
for kind in MODEL_KINDS:
    for layer, checkpoint in sorted(matched_runs[kind].items()):
        if checkpoint.is_file():
            geometry = load_projected_model(checkpoint, kind, results[layer]["pca"])
            cluster_geometry[kind][layer] = geometry
            invalid = int((~geometry["valid"]).sum())
            print(f"{kind} · layer {layer:02d}: loaded {checkpoint.name}; {invalid} invalid PCA footprints omitted", flush=True)
    print(f"{kind}: models loaded for {sorted(cluster_geometry[kind])}; remaining layers show points only.")


,layer,kind,K,threshold,available,run
0,4,hddc,100,0.1,True,hddc__temperature_02d__l04__k100__q16__s42__31...
2,4,kmeans,100,0.1,True,kmeans__temperature_02d__l04__k100__cattell__s...
1,28,hddc,100,0.1,True,hddc__temperature_02d__l28__k100__q16__s42__65...
3,28,kmeans,100,0.1,True,kmeans__temperature_02d__l28__k100__cattell__s...


kmeans · layer 04: loaded kmeans_model.pt; 1 invalid PCA footprints omitted
kmeans · layer 28: loaded kmeans_model.pt; 1 invalid PCA footprints omitted
kmeans: models loaded for [4, 28]; remaining layers show points only.
hddc · layer 04: loaded mfa_model.pt; 0 invalid PCA footprints omitted
hddc · layer 28: loaded mfa_model.pt; 0 invalid PCA footprints omitted
hddc: models loaded for [4, 28]; remaining layers show points only.


## Shared probability and geometry

`RETAINED_PROBABILITY` sets the initial probability. Each plot has its own slider
that resizes its geometry without loading models or refitting PCA. The radius is
`sqrt(chi2.ppf(p, r))`, where `r` is the number of nonzero **local PCs retained**
(three for HDDC; at most three for saved KMeans geometry). The probability belongs
to this truncated local Gaussian before projection, not to the full ambient
Gaussian or mixture. Projection may collapse directions; it does not change the
chosen local probability radius.

The colored segments are the projected local principal semiaxes:
**PC1 red, PC2 green, PC3 blue**. They retain their local ordering, even when the
global projection changes their apparent lengths or angles. `LOCAL_PC_COLORS`
controls these colors. Flat ellipses, lines, and points remain degenerate; no
jitter is added. Component IDs and surface colors are local to each model and
do not imply matching clusters across the two panels.


In [7]:
RETAINED_PROBABILITY = 0.90
ELLIPSE_ALPHA = 0.12
SHOW_CLUSTER_AXES = True
LOCAL_PC_COLORS = (0xE41A1C, 0x2CA02C, 0x377EB8)  # local PC1 red, PC2 green, PC3 blue


def validate_plot_request(pcs, dimension, probability):
    if (len(pcs) != dimension or len(set(pcs)) != dimension
            or any(not isinstance(pc, (int, np.integer)) or not 1 <= pc <= N_COMPONENTS for pc in pcs)):
        raise ValueError(f"Choose {dimension} distinct PCs from 1 to {N_COMPONENTS}")
    if not np.isfinite(probability) or not 0 < probability < 1:
        raise ValueError("Retained probability must be strictly between 0 and 1")
    if not 0 <= ELLIPSE_ALPHA <= 1:
        raise ValueError("ELLIPSE_ALPHA must be between 0 and 1")


def gaussian_footprint(mean, covariance, probability, probability_rank=None):
    """Project a local probability region using its retained local dimension."""
    if not np.isfinite(probability) or not 0 < probability < 1:
        raise ValueError("Retained probability must be strictly between 0 and 1")
    mean = np.asarray(mean, dtype=np.float64)
    covariance = np.asarray(covariance, dtype=np.float64)
    if not (np.isfinite(mean).all() and np.isfinite(covariance).all()):
        raise ValueError("Nonfinite Gaussian geometry")
    values, vectors = np.linalg.eigh((covariance + covariance.T) * 0.5)
    tolerance = max(float(np.max(np.abs(values))) * 1e-9, np.finfo(float).tiny)
    if values.min() < -tolerance:
        raise ValueError("Projected covariance has materially negative eigenvalues")
    keep = np.flatnonzero(values > tolerance)[::-1]
    rank = len(keep)
    if rank == 0:
        return {"rank": 0, "vertices": mean[None], "faces": None, "axes": []}
    degrees = rank if probability_rank is None else probability_rank
    if degrees < rank:
        raise ValueError("Local probability rank cannot be smaller than projected rank")
    transform = vectors[:, keep] * np.sqrt(values[keep] * chi2.ppf(probability, degrees))
    axes = [np.stack([mean - axis, mean + axis]) for axis in transform.T]
    faces = None
    if rank == 1:
        vertices = axes[0]
    elif rank == 2:
        angles = np.linspace(0, 2 * np.pi, 65)
        circle = np.stack([np.cos(angles), np.sin(angles)])
        vertices = mean + (transform @ circle).T
        if len(mean) == 3:
            # Convex planar polygon, triangulated without introducing thickness.
            faces = np.array([[0, j, j + 1] for j in range(1, len(vertices) - 2)], dtype=np.uint32)
    else:
        # Unique poles avoid zero-area triangles in the sphere mesh.
        n_u, n_v = 24, 12
        u = np.arange(n_u) * (2 * np.pi / n_u)
        v = np.arange(1, n_v) * (np.pi / n_v)
        uu, vv = np.meshgrid(u, v)
        rings = np.stack([np.cos(uu) * np.sin(vv), np.sin(uu) * np.sin(vv), np.cos(vv)], axis=-1).reshape(-1, 3)
        sphere = np.vstack([[0, 0, 1], rings, [0, 0, -1]])
        triangles = []
        for j in range(n_u):
            nxt = (j + 1) % n_u
            triangles.append([0, 1 + j, 1 + nxt])
            for ring in range(n_v - 2):
                a, b = 1 + ring * n_u + j, 1 + ring * n_u + nxt
                triangles.extend([[a, a + n_u, b], [b, a + n_u, b + n_u]])
            last = 1 + (n_v - 2) * n_u
            triangles.append([len(sphere) - 1, last + nxt, last + j])
        vertices = mean + sphere @ transform.T
        faces = np.asarray(triangles, dtype=np.uint32)
    return {"rank": rank, "vertices": vertices, "faces": faces, "axes": axes}


def layer_components(kind, layer, pcs, probability):
    geometry = cluster_geometry[kind].get(layer)
    if geometry is None:
        return []
    indices = np.asarray(pcs) - 1
    components = []
    for k, mean in enumerate(geometry["means"][:, indices]):
        covariance = geometry["covariances"][k][np.ix_(indices, indices)]
        local_rank = int(geometry["local_ranks"][k])
        footprint = gaussian_footprint(mean, covariance, probability, local_rank) if geometry["valid"][k] else None
        if footprint is not None:
            radius = np.sqrt(chi2.ppf(probability, local_rank)) if local_rank else 0.0
            projected_axes = geometry["local_pc_factors"][k, indices, :] * radius
            footprint["axes"] = [np.stack([mean - axis, mean + axis]) for axis in projected_axes.T]
        components.append({"id": k, "mean": mean, "color": geometry["colors"][k],
                           "hover": geometry["hover"][k], "footprint": footprint})
    return components


def projection_title(kind, layer, pcs, probability):
    geometry = cluster_geometry[kind].get(layer)
    note = "no model available" if geometry is None else (
        f"{geometry['description']} · K={len(geometry['means'])} · threshold={SURGERY_THRESHOLD:g}"
        f" · {int((~geometry['valid']).sum())} invalid footprints omitted")
    return f"{SUBSET} · layer {layer:02d} · PCs {tuple(pcs)}<br><sup>{note} · retained probability {probability:.1%}</sup>"


## Plot helpers

Each explorer has independent layer, PC, point-size, probability, and visibility
controls and its own camera. Both use the fitted activation PCA and ground-truth
colors. Run either plot cell below independently after preparing the models.

Uncheck **Show points** to see only the cluster geometry. Uncheck **Show clusters**
to inspect points alone. Missing models are reported for the selected layer.

Click an ellipsoid to hide that cluster, including its axes and center. Hidden
clusters stay hidden when changing PCs or probability and are remembered per
layer within each plot. **Restore hidden clusters** restores the current layer.


In [8]:
import ipywidgets as widgets


def k3d_cluster_objects(kind, layer, pcs, probability, hidden, on_hide):
    components = [c for c in layer_components(kind, layer, pcs, probability)
                  if c["id"] not in hidden]
    objects = []
    scores = results[layer]["scores"][:, np.asarray(pcs) - 1]
    span = max(float(np.ptp(scores, axis=0).max()), 1e-6)
    for component in components:
        footprint = component["footprint"]
        if footprint is None:
            continue
        vertices = footprint["vertices"].astype(np.float32)
        color = int(component["color"])
        name = component["hover"].replace("<br>", " · ")
        if footprint["faces"] is not None:
            obj = k3d.mesh(vertices, footprint["faces"], color=color, opacity=ELLIPSE_ALPHA,
                           side="double", name=name)
        elif footprint["rank"] == 0:
            obj = k3d.points(vertices, color=color, point_size=span * 0.005, name=name)
        else:
            obj = k3d.line(vertices, color=color, width=span * 0.001, name=name)
        if footprint["faces"] is not None or footprint["rank"] == 0:
            obj.click_callback = lambda event, component_id=component["id"]: on_hide(layer, component_id)
        objects.append(obj)
    if SHOW_CLUSTER_AXES:
        for pc, color in enumerate(LOCAL_PC_COLORS):
            segments = [c["footprint"]["axes"][pc] for c in components
                        if c["footprint"] is not None
                        and np.linalg.norm(c["footprint"]["axes"][pc][1] - c["footprint"]["axes"][pc][0]) > 0]
            if segments:
                vertices = np.concatenate(segments).astype(np.float32)
                indices = np.arange(len(vertices), dtype=np.uint32).reshape(-1, 2)
                objects.append(k3d.lines(vertices, indices, indices_type="segment", color=color,
                                         width=span * 0.0012, name=f"Local PC{pc + 1}"))
    if components:
        objects.append(k3d.points(np.stack([c["mean"] for c in components]).astype(np.float32),
                                 colors=np.array([int(c["color"]) for c in components], dtype=np.uint32),
                                 point_size=span * 0.008, name="Component centers"))
    return objects


def make_cluster_explorer(kind):
    """Create one plot with its own controls, camera, and overlay objects."""
    validate_plot_request((1, 2, 3), 3, RETAINED_PROBABILITY)
    k3d_colors = pd.to_numeric(labels[color_label], errors="raise").to_numpy(dtype=np.float32)
    if not np.isfinite(k3d_colors).all():
        raise ValueError("K3D coloring requires a finite numeric COLOR_LABEL")
    modeled_layers = [layer for layer in LAYERS if layer in cluster_geometry[kind]]
    initial_layer = (modeled_layers or LAYERS)[0]
    layer_control = widgets.Dropdown(options=LAYERS, value=initial_layer, description="Layer")
    pc_options = list(range(1, results[LAYERS[0]]["scores"].shape[1] + 1))
    pc_controls = [widgets.Dropdown(options=pc_options, value=i + 1, description=axis)
                   for i, axis in enumerate(["X PC", "Y PC", "Z PC"])]
    size_control = widgets.FloatLogSlider(value=0.003, base=10, min=-4, max=-1,
                                         step=0.1, description="Point size", continuous_update=False)
    probability_control = widgets.FloatSlider(value=RETAINED_PROBABILITY, min=min(0.01, RETAINED_PROBABILITY),
                                              max=max(0.999, RETAINED_PROBABILITY), step=0.001,
                                              readout_format=".3f", description="Probability", continuous_update=False)
    overlay_control = widgets.Checkbox(value=True, description="Show clusters")
    points_control = widgets.Checkbox(value=True, description="Show points")
    restore_button = widgets.Button(description="Restore hidden clusters", disabled=True,
                                    layout=widgets.Layout(width="200px"))
    hidden_components = {layer: set() for layer in LAYERS}
    status = widgets.HTML()
    initial = np.asarray(results[initial_layer]["scores"][:, :3], dtype=np.float32)
    cloud = k3d.points(initial.copy(), attribute=k3d_colors,
                       color_map=k3d.colormaps.matplotlib_color_maps.Viridis,
                       color_range=[float(k3d_colors.min()), float(k3d_colors.max())],
                       shader="3d", opacity=1.0, name=color_label,
                       point_size=max(float(np.ptp(initial, axis=0).max()) * size_control.value, 1e-6))
    plot = k3d.plot(height=720, camera_auto_fit=False, mode="callback")
    plot += cloud
    plot.colorbar_object_id = cloud.id
    objects = []
    panel_status = widgets.HTML()


    def hide_component(layer, component_id):
        hidden_components[layer].add(component_id)
        update_k3d({"owner": restore_button})

    def restore_clusters(button):
        hidden_components[layer_control.value].clear()
        update_k3d({"owner": button})

    def update_k3d(change=None):
        nonlocal plot
        pcs = tuple(control.value for control in pc_controls)
        if len(set(pcs)) != 3:
            status.value = "Choose three distinct PCs; the previous projections are still displayed."
            return
        probability = probability_control.value
        validate_plot_request(pcs, 3, probability)
        layer = layer_control.value
        indices = np.array(pcs) - 1
        scores = np.asarray(results[layer]["scores"][:, indices], dtype=np.float32)
        ratios = results[layer]["pca"].explained_variance_ratio_[indices]
        point_size = max(float(np.ptp(scores, axis=0).max()) * size_control.value, 1e-6)
        status.value = f"Layer {layer:02d} · PCs {pcs} · {ratios.sum():.1%} explained · {len(scores):,} prompts"
        hidden = hidden_components[layer]
        status.value += f" · {len(hidden)} hidden clusters"
        restore_button.disabled = not hidden
        owner = change["owner"] if change is not None else None
        cloud.positions = scores
        cloud.point_size = point_size
        cloud.visible = points_control.value
        plot.axes = [f"PC{pc} ({ratio:.1%})" for pc, ratio in zip(pcs, ratios)]
        panel_status.value = projection_title(kind, layer, pcs, probability).replace("<sup>", "").replace("</sup>", "")
        if owner is overlay_control:
            for obj in objects:
                obj.visible = overlay_control.value
        elif owner is not size_control and owner is not points_control:
            # Replace the old geometry when the projection changes.
            for obj in objects:
                plot -= obj
            objects.clear()
            for obj in k3d_cluster_objects(kind, layer, pcs, probability, hidden, hide_component):
                obj.visible = overlay_control.value
                plot += obj
                objects.append(obj)
        if owner is None or owner is layer_control or any(owner is control for control in pc_controls):
            plot.camera_reset()


    for control in [layer_control, *pc_controls, size_control, probability_control, overlay_control, points_control]:
        control.observe(update_k3d, names="value")
    restore_button.on_click(restore_clusters)
    update_k3d()
    return widgets.VBox([
        widgets.HBox([layer_control, *pc_controls]),
        widgets.HBox([size_control, probability_control]),
        widgets.HBox([points_control, overlay_control, restore_button]),
        widgets.HTML("Local covariance axes: " + " &nbsp; ".join(
            f'<span style="color:#{color:06x}"><b>PC{pc + 1}</b></span>'
            for pc, color in enumerate(LOCAL_PC_COLORS))),
        status, panel_status, plot,
    ], layout=widgets.Layout(width="100%"))


## KMeans plot


In [9]:
kmeans_explorer = make_cluster_explorer("kmeans")
display(kmeans_explorer)


## HDDC plot


In [10]:
hddc_explorer = make_cluster_explorer("hddc")
display(hddc_explorer)
